In [7]:
import pandas as pd

columns = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

columns += [f"Wilderness_Area{i}" for i in range(1, 5)]
columns += [f"Soil_Type{i}" for i in range(1, 41)]
columns += ["Cover_Type"]

df = pd.read_csv("/content/covtype.data.gz", header=None)
df.columns = columns

print(df.shape)
print(df["Cover_Type"].value_counts().sort_index())

(581012, 55)
Cover_Type
1    211840
2    283301
3     35754
4      2747
5      9493
6     17367
7     20510
Name: count, dtype: int64


In [8]:
# split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["Cover_Type"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,
    stratify=temp_df["Cover_Type"],
    random_state=42
)

print(train_df.shape, val_df.shape, test_df.shape)

# balanced training set
train_balanced = (
    train_df.groupby("Cover_Type", group_keys=False)
    .apply(lambda x: x.sample(n=4000, replace=len(x) < 4000, random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# keep validation/test realistic
val_small = val_df.sample(n=5000, random_state=42).reset_index(drop=True)
test_small = test_df.sample(n=5000, random_state=42).reset_index(drop=True)

print(train_balanced["Cover_Type"].value_counts().sort_index())

(406708, 55) (58101, 55) (116203, 55)
Cover_Type
1    4000
2    4000
3    4000
4    4000
5    4000
6    4000
7    4000
Name: count, dtype: int64


/tmp/ipykernel_12902/2668854213.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=4000, replace=len(x) < 4000, random_state=42))


In [9]:
print(df.columns.tolist())

['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4', 'Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 'Soil_Type15', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40', 'Cover_Type']


In [10]:
print("Original:")
print(df["Cover_Type"].value_counts(normalize=True).sort_index())

print("\nTrain:")
print(train_df["Cover_Type"].value_counts(normalize=True).sort_index())

print("\nValidation:")
print(val_df["Cover_Type"].value_counts(normalize=True).sort_index())

print("\nTest:")
print(test_df["Cover_Type"].value_counts(normalize=True).sort_index())

Original:
Cover_Type
1    0.364605
2    0.487599
3    0.061537
4    0.004728
5    0.016339
6    0.029891
7    0.035300
Name: proportion, dtype: float64

Train:
Cover_Type
1    0.364606
2    0.487598
3    0.061538
4    0.004728
5    0.016339
6    0.029891
7    0.035301
Name: proportion, dtype: float64

Validation:
Cover_Type
1    0.364606
2    0.487599
3    0.061531
4    0.004733
5    0.016334
6    0.029896
7    0.035301
Name: proportion, dtype: float64

Test:
Cover_Type
1    0.364603
2    0.487604
3    0.061539
4    0.004724
5    0.016342
6    0.029887
7    0.035300
Name: proportion, dtype: float64


In [11]:
wilderness_cols = [f"Wilderness_Area{i}" for i in range(1, 5)]
soil_cols = [f"Soil_Type{i}" for i in range(1, 41)]

def get_active_index(row, cols):
    for i, col in enumerate(cols, start=1):
        if row[col] == 1:
            return i
    return 0

def row_to_prompt(row):
    wilderness = get_active_index(row, wilderness_cols)
    soil = get_active_index(row, soil_cols)

    return f"""Task: Predict the forest cover type.

Valid labels:
<C1>, <C2>, <C3>, <C4>, <C5>, <C6>, <C7>

Now classify this:

Features:
Elevation={row['Elevation']}
Aspect={row['Aspect']}
Slope={row['Slope']}
Horizontal_Distance_To_Hydrology={row['Horizontal_Distance_To_Hydrology']}
Vertical_Distance_To_Hydrology={row['Vertical_Distance_To_Hydrology']}
Horizontal_Distance_To_Roadways={row['Horizontal_Distance_To_Roadways']}
Hillshade_9am={row['Hillshade_9am']}
Hillshade_Noon={row['Hillshade_Noon']}
Hillshade_3pm={row['Hillshade_3pm']}
Horizontal_Distance_To_Fire_Points={row['Horizontal_Distance_To_Fire_Points']}
Wilderness_Area={wilderness}
Soil_Type={soil}

Rules:
- Output exactly one label
- Do not explain

Answer:"""

def label_to_token(label):
    return f"<C{int(label)}>"

In [12]:
# building dataset
import torch
from torch.utils.data import Dataset

class CovertypeLLMDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=224):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = row_to_prompt(row)
        label_text = " " + label_to_token(row["Cover_Type"])
        full_text = prompt + label_text

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = full_enc["input_ids"].squeeze(0)
        attention_mask = full_enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_len = int(prompt_enc["attention_mask"].sum().item())

        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [13]:
# loading decoder only
!pip install transformers --upgrade # Add this line to ensure the library is up to date and correctly installed.
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

special_tokens = {
    "additional_special_tokens": [f"<C{i}>" for i in range(1, 8)]
}
tokenizer.add_special_tokens(special_tokens)

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

# ✅ ADD THIS
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [14]:
# train/val/test

train_dataset = CovertypeLLMDataset(train_balanced, tokenizer, max_length=224)
val_dataset = CovertypeLLMDataset(val_small, tokenizer, max_length=224)
test_dataset = CovertypeLLMDataset(test_small, tokenizer, max_length=224)

print("train size:", len(train_dataset))
print("val size:", len(val_dataset))

sample = train_dataset[0]
print("Labels not masked:", (sample["labels"] != -100).sum().item())

train size: 28000
val size: 5000
Labels not masked: 2


In [15]:
# 🔍 check prompt length
row = train_balanced.iloc[0]
prompt = row_to_prompt(row)

print(prompt)
print("Token length:", len(tokenizer(prompt)["input_ids"]))

Task: Predict the forest cover type.

Valid labels:
<C1>, <C2>, <C3>, <C4>, <C5>, <C6>, <C7>

Now classify this:

Features:
Elevation=2745
Aspect=102
Slope=15
Horizontal_Distance_To_Hydrology=42
Vertical_Distance_To_Hydrology=10
Horizontal_Distance_To_Roadways=1591
Hillshade_9am=244
Hillshade_Noon=219
Hillshade_3pm=100
Horizontal_Distance_To_Fire_Points=2546
Wilderness_Area=1
Soil_Type=23

Rules:
- Output exactly one label
- Do not explain

Answer:
Token length: 171


In [16]:
print("df:", "Cover_Type" in df.columns)
print("train_df:", "Cover_Type" in train_df.columns)
print(train_df.columns.tolist()[-5:])

df: True
train_df: True
['Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40', 'Cover_Type']


In [ ]:
# training
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
import numpy as np
import torch

# metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    # ignore masked tokens (-100)
    mask = labels != -100
    labels = labels[mask]
    predictions = predictions[mask]

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

# training arguments
training_args = TrainingArguments(
    output_dir="./llm_covertype",

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=10,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,

    eval_accumulation_steps=16,

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,  # ✅ important for custom dataset

    report_to="none"
)

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# start training
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


In [ ]:
# prediction
def predict_label(row, model, tokenizer, max_new_tokens=5, device="cpu"):
    prompt = row_to_prompt(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=224
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_ids = output_ids[0][input_len:]
    decoded = tokenizer.decode(generated_ids, skip_special_tokens=False)

    for i in range(1, 8):
        if f"<C{i}>" in decoded:
            return i

    return None

In [ ]:
# eval
from sklearn.metrics import classification_report, accuracy_score
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

y_true = []
y_pred = []

valid_labels = [1, 2, 3, 4, 5, 6, 7]

sample_test_df = test_small

for _, row in sample_test_df.iterrows():
    pred = predict_label(row, model, tokenizer, device=device)

    if pred not in valid_labels:
        continue

    y_true.append(int(row["Cover_Type"]))
    y_pred.append(int(pred))

print("Number of valid predictions:", len(y_pred))

if len(y_pred) > 0:
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, labels=valid_labels, digits=4))
else:
    print("No valid predictions were generated.")

In [ ]:
import sys
print(sys.executable)

import transformers
print(transformers.__version__)